# Bronze Data Quality & Consistency Check
Notebook này dùng để kiểm tra tính nhất quán giữa dữ liệu log raw (`JSONL`) sinh ra từ Simulator và dữ liệu đã ingest vào `Delta Bronze` trên MinIO.

**Các bước check độ tối ưu (Reconciliation) cho Data Pipeline:**
1. **Row Count Check:** Đếm số dòng raw (sau khi áp dụng rule lọc) và so sánh với tổng số dòng trong Delta.
2. **Uniqueness Check:** Đảm bảo không có ID bị trùng lặp trong Delta (tránh trường hợp duplicate khi append/merge sai cách).
3. **Data Loss Check (Anti-Join):** Dùng `subtract()` để tìm tập các record có trong raw hợp lệ nhưng lại rơi rớt mất khi ghi vào Delta.
4. **Metric Consistency:** Aggregate một chỉ số quan trọng (vd: tổng `fare_vnd`) nhằm đảm bảo không có sự toàn vẹn nào bị phá vỡ giữa raw và bronze.

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as _sum

# Cấu hình Access MinIO
MINIO_ENDPOINT = "http://minio:9000"
MINIO_KEY = "minioadmin"
MINIO_SECRET = "minioadmin123"

# Khởi tạo Spark gắn với Delta & MinIO để đọc cả Raw (JSONL) và Bronze (Delta)
spark = (
    SparkSession.builder
    .appName("rideflow_data_check")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("✅ Spark Session cho Data Check đã sẵn sàng!")

✅ Spark Session cho Data Check đã sẵn sàng!


In [5]:
import os
from pyspark.sql.functions import col, lit

# Cấu hình rule lọc và đường dẫn cho từng bảng
TABLES_CONFIG = {
    "trips": {
        "pk": "trip_id",
        "raw_dir": os.path.abspath("../spark-data/raw/trips"),
        "bronze_path": "s3a://rideflow/bronze/trips",
        "filter": lambda df: (
            df.dropDuplicates(["trip_id"])
              .filter(col("status").isin(["completed", "cancelled", "no_driver"]))
              .filter(col("city").isin(["HCMC", "HANOI"]))
              .filter(col("trip_id").isNotNull() & col("rider_id").isNotNull())
        )
    },
    "payments": {
        "pk": "payment_id",
        "raw_dir": os.path.abspath("../spark-data/raw/payments"),
        "bronze_path": "s3a://rideflow/bronze/payments",
        "filter": lambda df: (
            df.dropDuplicates(["payment_id"])
              .filter(col("payment_status").isin(["success", "refunded", "failed"]))
              .filter(col("payment_id").isNotNull() & col("trip_id").isNotNull())
        )
    },
    "ratings": {
        "pk": "rating_id",
        "raw_dir": os.path.abspath("../spark-data/raw/ratings"),
        "bronze_path": "s3a://rideflow/bronze/ratings",
        "filter": lambda df: (
            df.dropDuplicates(["rating_id"])
              .filter(col("stars").between(1, 5))
              .filter(col("rating_id").isNotNull() & col("trip_id").isNotNull())
        )
    }
}

print("=====================================================")
print("📊 BẮT ĐẦU ĐỐI CHIẾU DỮ LIỆU RAW VÀ BRONZE TẤT CẢ BẢNG")
print("=====================================================\n")

for table_name, config in TABLES_CONFIG.items():
    print(f"📌 Đang kiểm tra bảng: {table_name.upper()}")
    raw_base_dir = config["raw_dir"]
    
    # Lấy danh sách các ngày phân vùng
    partitions = []
    if os.path.exists(raw_base_dir):
        for d in os.listdir(raw_base_dir):
            if d.startswith("date="):
                partitions.append(d.split("date=")[1])
    partitions.sort()
    
    if not partitions:
        print(f"  -> Không tìm thấy dữ liệu Raw tại {raw_base_dir}\n")
        continue
        
    print(f"  -> Tìm thấy {len(partitions)} ngày trong Raw data.")
    
    mismatched_dates = []
    
    for date in partitions:
        raw_path = os.path.join(raw_base_dir, f"date={date}", "data.jsonl")
        
        try:
            raw_df = spark.read.json(raw_path)
            # Apply filter logic
            raw_valid_df = config["filter"](raw_df)
            raw_valid_count = raw_valid_df.count()
        except Exception as e:
            print(f"  ⚠️ Bỏ qua ngày {date}: Lỗi đọc Raw ({e})")
            continue
            
        bronze_path = config["bronze_path"]
        try:
            bronze_df = spark.read.format("delta").load(bronze_path).filter(col("ingest_date") == date)
            bronze_count = bronze_df.count()
        except Exception as e:
            bronze_count = 0
            
        if raw_valid_count != bronze_count:
            mismatched_dates.append({
                "date": date,
                "raw": raw_valid_count,
                "bronze": bronze_count
            })
            
    if not mismatched_dates:
        print(f"  => ✅ HOÀN HẢO! Tất cả {len(partitions)} ngày đều khớp số lượng dòng giữa Raw và MinIO.\n")
    else:
        print(f"  => ⚠️ Phát hiện {len(mismatched_dates)} ngày bị lệch dữ liệu:")
        for m in mismatched_dates:
            print(f"     * Ngày {m['date']}: Raw {m['raw']} dòng | Bronze {m['bronze']} dòng | Lệch {m['raw'] - m['bronze']} dòng")
        print("\n")

print("="*60)
print("🏁 ĐÃ HOÀN TẤT KIỂM TRA TOÀN BỘ CÁC BẢNG")
print("="*60)


📊 BẮT ĐẦU ĐỐI CHIẾU DỮ LIỆU RAW VÀ BRONZE TẤT CẢ BẢNG

📌 Đang kiểm tra bảng: TRIPS
  -> Tìm thấy 3 ngày trong Raw data.


  => ✅ HOÀN HẢO! Tất cả 3 ngày đều khớp số lượng dòng giữa Raw và MinIO.

📌 Đang kiểm tra bảng: PAYMENTS
  -> Tìm thấy 3 ngày trong Raw data.


  => ✅ HOÀN HẢO! Tất cả 3 ngày đều khớp số lượng dòng giữa Raw và MinIO.

📌 Đang kiểm tra bảng: RATINGS
  -> Tìm thấy 3 ngày trong Raw data.


  => ✅ HOÀN HẢO! Tất cả 3 ngày đều khớp số lượng dòng giữa Raw và MinIO.

🏁 ĐÃ HOÀN TẤT KIỂM TRA TOÀN BỘ CÁC BẢNG


In [6]:
# =====================================================================
# ĐOẠN CODE ĐỂ KIỂM TRA CHI TIẾT SỰ LỆCH/KHÁC BIỆT CHO 1 NGÀY & 1 BẢNG
# =====================================================================
TARGET_TABLE = "trips"      # Cần thay đổi: "trips", "payments" hoặc "ratings"
TARGET_DATE = "2026-03-24"  # Cần thay đổi: Ngày bị lệch (VD: "2026-03-24")

print(f"🔍 TÌM KIẾM DỮ LIỆU LỆCH CHO BẢNG {TARGET_TABLE.upper()} - NGÀY {TARGET_DATE}\n")

if TARGET_TABLE not in TABLES_CONFIG:
    print("❌ TARGET_TABLE không hợp lệ! Hãy chọn 'trips', 'payments', hoặc 'ratings'")
else:
    config = TABLES_CONFIG[TARGET_TABLE]
    pk = config["pk"]
    
    raw_path = os.path.join(config["raw_dir"], f"date={TARGET_DATE}", "data.jsonl")
    bronze_path = config["bronze_path"]
    
    # 1. Đọc và chuẩn bị Raw
    try:
        raw_df = spark.read.json(raw_path)
        raw_valid_df = config["filter"](raw_df)
        
        # 2. Đọc Bronze
        bronze_df = spark.read.format("delta").load(bronze_path).filter(col("ingest_date") == TARGET_DATE)
        
        # 3. Mất khi lưu (Có ở Raw hợp lệ, Không có ở Bronze)
        missing_in_bronze = raw_valid_df.select(pk).subtract(bronze_df.select(pk))
        missing_count = missing_in_bronze.count()
        
        print(f"❌ Số lượng dòng hợp lệ bị mất khi lưu vào MinIO: {missing_count}")
        if missing_count > 0:
            print("   -> Mẫu dữ liệu bị mất (tối đa 10 dòng):")
            raw_valid_df.join(missing_in_bronze, on=pk, how="inner").show(10, truncate=False)
            
        # 4. Rác thừa (Có ở Bronze, Không có ở Raw hợp lệ)
        extra_in_bronze = bronze_df.select(pk).subtract(raw_valid_df.select(pk))
        extra_count = extra_in_bronze.count()
        
        print(f"\n❓ Số lượng dòng có mặt trên MinIO nhưng không có/hợp lệ trong Raw: {extra_count}")
        if extra_count > 0:
            print("   -> Mẫu dữ liệu dư thừa trên MinIO (tối đa 10 dòng):")
            bronze_df.join(extra_in_bronze, on=pk, how="inner").show(10, truncate=False)
            
    except Exception as e:
        print(f"❌ Lỗi truy xuất dữ liệu: {e}")


🔍 TÌM KIẾM DỮ LIỆU LỆCH CHO BẢNG TRIPS - NGÀY 2026-03-24



❌ Số lượng dòng hợp lệ bị mất khi lưu vào MinIO: 0



❓ Số lượng dòng có mặt trên MinIO nhưng không có/hợp lệ trong Raw: 0
